In [5]:
import pandas as pd
import os

def aggregate_calories_by_day(file_path='hourlyCalories_merged.csv'):
    """
    Aggregates hourly calorie data into daily totals.
    
    Parameters:
    -----------
    file_path : str
        Path to the hourly calories CSV file
        
    Returns:
    --------
    pandas.DataFrame
        DataFrame with daily calorie totals
    """
    # Read the hourly data
    df = pd.read_csv(file_path)
    
    # Convert ActivityHour to datetime
    df['ActivityHour'] = pd.to_datetime(df['ActivityHour'])
    
    # Extract date (without time)
    df['Date'] = df['ActivityHour'].dt.date
    
    # Group by ID and Date, summing calories
    daily_calories = df.groupby(['Id', 'Date'])['Calories'].sum().reset_index()
    
    # Rename columns for clarity
    daily_calories.rename(columns={'Calories': 'TotalCalories'}, inplace=True)
    
    print(f"Successfully aggregated data for {len(daily_calories)} day-user combinations")
    
    # Save to CSV
    output_path = 'dailyCalories_aggregated.csv'
    daily_calories.to_csv(output_path, index=False)
    print(f"Output saved to {os.path.abspath(output_path)}")
    
    return daily_calories

# Example usage
if __name__ == "__main__":
    daily_data = aggregate_calories_by_day("/home/bshumway9/CS-4320/project/archive/3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/hourlyCalories_merged.csv")
    
    # Display summary statistics
    print("\nSummary statistics:")
    print(daily_data['TotalCalories'].describe())
    
    # Show the first few rows
    print("\nSample of aggregated data:")
    print(daily_data.head())

In [8]:
import pandas as pd
import os

# Read the CSV data
df = pd.read_csv('/home/bshumway9/CS-4320/project/archive/4.12.16-5.12.16/Fitabase Data 4.12.16-5.12.16/minuteSleep_merged.csv')

# Convert date column to datetime 
df['date'] = pd.to_datetime(df['date'])

# Extract date part only (without time)
df['day'] = df['date'].dt.date

# Group by Id and day
daily_sleep = df.groupby(['Id', 'day']).agg(
    sleep_minutes=('value', 'count'),  # Count minutes of sleep
    avg_sleep_quality=('value', 'mean'),  # Average sleep quality (1-3)
    sleep_records=('logId', 'nunique')  # Number of sleep sessions
).reset_index()

# Format sleep duration as hours and minutes
daily_sleep['sleep_duration'] = daily_sleep['sleep_minutes'].apply(
    lambda x: f"{x//60}h {x%60}m"
)

# Format the day as a string
daily_sleep['day'] = daily_sleep['day'].astype(str)

# Sort by Id and day
daily_sleep = daily_sleep.sort_values(['Id', 'day'])

# Save to CSV file
output_path = 'dailySleep_aggregated.csv'
daily_sleep.to_csv(output_path, index=False)
print(f"Successfully aggregated sleep data for {len(daily_sleep)} day-user combinations")
print(f"Output saved to {os.path.abspath(output_path)}")

# Display the result
print("\nSample of aggregated data:")
print(daily_sleep[['Id', 'day', 'sleep_duration', 'sleep_minutes', 'avg_sleep_quality', 'sleep_records']].head())

In [2]:
import pandas as pd
from datetime import datetime

def aggregate_high_intensity_data(input_file, output_file):
    """
    Aggregates minute-level intensity data to daily totals, counting only intensities of 2 or 3.
    Each minute with intensity of 2 or 3 contributes exactly 1 to the total intensity.
    
    Args:
        input_file (str): Path to minuteIntensitiesNarrow_merged.csv
        output_file (str): Path to save the aggregated data
    """
    # Read the input file
    df = pd.read_csv(input_file)
    
    # Convert ActivityMinute to datetime
    df['ActivityMinute'] = pd.to_datetime(df['ActivityMinute'])
    
    # Extract date from datetime
    df['Date'] = df['ActivityMinute'].dt.date
    
    # Filter for intensities of 2 or 3 only
    high_intensity_df = df[df['Intensity'].isin([2, 3])]
    
    # Count the number of minutes with high intensity per user per day
    # Each minute with intensity 2 or 3 contributes exactly 1 to the total
    daily_intensity = high_intensity_df.groupby(['Id', 'Date']).size().reset_index(name='TotalIntensity')
    
    # Convert Date back to string format for CSV
    daily_intensity['Date'] = daily_intensity['Date'].astype(str)
    
    # Save to output file
    daily_intensity.to_csv(output_file, index=False)
    
    print(f"Processed {len(df)} records")
    print(f"Found {len(high_intensity_df)} high intensity minutes")
    print(f"Aggregated to {len(daily_intensity)} daily records")
    print(f"Output saved to {output_file}")
    
    return daily_intensity

# Example usage:
aggregate_high_intensity_data(
    '/home/bshumway9/CS-4320/project/archive/4.12.16-5.12.16/Fitabase Data 4.12.16-5.12.16/minuteIntensitiesNarrow_merged.csv', 
    '/home/bshumway9/CS-4320/project/archive/4.12.16-5.12.16/Fitabase Data 4.12.16-5.12.16/dailyHighIntensities_aggregated.csv'
)

/tmp/ipykernel_370904/3731004193.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['ActivityMinute'] = pd.to_datetime(df['ActivityMinute'])


Processed 1325580 records
Found 32587 high intensity minutes
Aggregated to 559 daily records
Output saved to /home/bshumway9/CS-4320/project/archive/4.12.16-5.12.16/Fitabase Data 4.12.16-5.12.16/dailyHighIntensities_aggregated.csv


,Id,Date,TotalIntensity
0,1503960366,2016-04-12,38
1,1503960366,2016-04-13,40
2,1503960366,2016-04-14,41
3,1503960366,2016-04-15,63
4,1503960366,2016-04-16,46
...,...,...,...
554,8877689391,2016-05-08,21
555,8877689391,2016-05-09,92
556,8877689391,2016-05-10,29
557,8877689391,2016-05-11,100


In [13]:
import pandas as pd
import os
from datetime import datetime

def aggregate_hourly_to_daily_intensities(file_path='hourlyIntensities_merged.csv'):
    """
    Aggregates hourly intensity data into daily summaries.
    
    Parameters:
    -----------
    file_path : str
        Path to the hourly intensities CSV file
        
    Returns:
    --------
    pandas.DataFrame
        DataFrame with daily intensity totals
    """
    # Read the hourly data
    df = pd.read_csv(file_path)
    
    # Convert ActivityHour to datetime
    df['ActivityHour'] = pd.to_datetime(df['ActivityHour'])
    
    # Extract date (without time)
    df['ActivityDay'] = df['ActivityHour'].dt.date
    
    # Group by Id and ActivityDay
    daily_intensities = df.groupby(['Id', 'ActivityDay']).agg(
        TotalIntensity=('TotalIntensity', 'sum'),
        AverageIntensity=('AverageIntensity', 'mean')
    ).reset_index()
    
    # Calculate hours with activity (where TotalIntensity > 0)
    hours_with_activity = df[df['TotalIntensity'] > 0].groupby(['Id', 'ActivityDay']).size().reset_index(name='HoursActive')
    
    # Merge with the main dataframe
    daily_intensities = daily_intensities.merge(hours_with_activity, on=['Id', 'ActivityDay'], how='left')
    daily_intensities['HoursActive'] = daily_intensities['HoursActive'].fillna(0)
    
    # Calculate max intensity hour
    max_intensity = df.loc[df.groupby(['Id', 'ActivityDay'])['TotalIntensity'].idxmax()]
    max_intensity = max_intensity[['Id', 'ActivityDay', 'TotalIntensity', 'ActivityHour']]
    max_intensity.rename(columns={'TotalIntensity': 'MaxIntensity', 'ActivityHour': 'MaxIntensityHour'}, inplace=True)
    
    # Merge with the main dataframe
    daily_intensities = daily_intensities.merge(max_intensity[['Id', 'ActivityDay', 'MaxIntensity', 'MaxIntensityHour']], 
                                               on=['Id', 'ActivityDay'], how='left')
    
    # Format the ActivityDay as string in MM/DD/YYYY format
    daily_intensities['ActivityDay'] = daily_intensities['ActivityDay'].apply(
        lambda x: datetime.strftime(x, '%m/%d/%Y')
    )
    
    # Format MaxIntensityHour as a time string (if needed)
    daily_intensities['MaxIntensityHour'] = daily_intensities['MaxIntensityHour'].dt.strftime('%I:%M %p')
    
    # Save to CSV
    output_path = 'dailyIntensities_aggregated.csv'
    daily_intensities.to_csv(output_path, index=False)
    print(f"Successfully aggregated data for {len(daily_intensities)} day-user combinations")
    print(f"Output saved to {os.path.abspath(output_path)}")
    
    return daily_intensities

# Example usage
if __name__ == "__main__":
    daily_data = aggregate_hourly_to_daily_intensities("/home/bshumway9/CS-4320/project/archive/4.12.16-5.12.16/Fitabase Data 4.12.16-5.12.16/hourlyIntensities_merged.csv")
    
    # Display summary statistics
    print("\nSummary statistics:")
    print(daily_data[['TotalIntensity', 'AverageIntensity', 'HoursActive']].describe())
    
    # Show the first few rows
    print("\nSample of aggregated data:")
    print(daily_data.head())

In [2]:
import pandas as pd
import os
from datetime import datetime

def aggregate_minute_to_daily_steps(file_path='minuteStepsNarrow_merged.csv'):
    """
    Aggregates minute-by-minute step data into daily summaries.
    
    Parameters:
    -----------
    file_path : str
        Path to the minute steps CSV file
        
    Returns:
    --------
    pandas.DataFrame
        DataFrame with daily step totals and related metrics
    """
    print(f"Reading minute step data from: {file_path}")
    # Read the minute data
    df = pd.read_csv(file_path)
    
    # Convert timestamp to datetime
    df['ActivityMinute'] = pd.to_datetime(df['ActivityMinute'])
    
    # Extract date (without time)
    df['ActivityDay'] = df['ActivityMinute'].dt.date
    
    # Group by Id and ActivityDay
    daily_steps = df.groupby(['Id', 'ActivityDay']).agg(
        TotalSteps=('Steps', 'sum'),
        MaxStepsMinute=('Steps', 'max'),
        ActiveMinutes=('Steps', lambda x: sum(x > 0))
    ).reset_index()
    
    # Calculate additional metrics
    daily_steps['AvgStepsPerActiveMinute'] = daily_steps.apply(
        lambda row: row['TotalSteps'] / row['ActiveMinutes'] if row['ActiveMinutes'] > 0 else 0, 
        axis=1
    )
    
    # Find peak activity times
    # Group by minute of the day and sum steps
    df['TimeOfDay'] = df['ActivityMinute'].dt.hour * 60 + df['ActivityMinute'].dt.minute
    peak_times = df.groupby(['Id', 'ActivityDay', 'TimeOfDay'])['Steps'].sum().reset_index()
    
    # Get the time with maximum steps for each day
    peak_times = peak_times.loc[peak_times.groupby(['Id', 'ActivityDay'])['Steps'].idxmax()]
    peak_times['PeakActivityTime'] = peak_times['TimeOfDay'].apply(
        lambda x: f"{x // 60:02d}:{x % 60:02d}"
    )
    
    # Merge with the main dataframe
    daily_steps = daily_steps.merge(
        peak_times[['Id', 'ActivityDay', 'PeakActivityTime']], 
        on=['Id', 'ActivityDay'], 
        how='left'
    )
    
    # Format the ActivityDay as string in MM/DD/YYYY format
    daily_steps['ActivityDay'] = daily_steps['ActivityDay'].apply(
        lambda x: datetime.strftime(x, '%m/%d/%Y')
    )
    
    # Round decimal values for readability
    daily_steps['AvgStepsPerActiveMinute'] = daily_steps['AvgStepsPerActiveMinute'].round(2)
    
    # Save to CSV
    output_path = 'dailySteps_aggregated.csv'
    daily_steps.to_csv(output_path, index=False)
    print(f"Successfully aggregated data for {len(daily_steps)} day-user combinations")
    print(f"Output saved to {os.path.abspath(output_path)}")
    
    return daily_steps

# Example usage
if __name__ == "__main__":
    daily_data = aggregate_minute_to_daily_steps("/home/bshumway9/CS-4320/project/archive/4.12.16-5.12.16/Fitabase Data 4.12.16-5.12.16/minuteStepsNarrow_merged.csv")
    
    # Display summary statistics
    print("\nSummary statistics:")
    print(daily_data[['TotalSteps', 'MaxStepsMinute', 'ActiveMinutes']].describe())
    
    # Show the first few rows
    print("\nSample of aggregated data:")
    print(daily_data.head())

In [4]:
import pandas as pd
import os
from datetime import datetime

def aggregate_heartrate_to_hourly(file_path='heartrate_seconds_merged.csv'):
    """
    Aggregates second-by-second heart rate data into hourly summaries.
    
    Parameters:
    -----------
    file_path : str
        Path to the heart rate seconds CSV file
        
    Returns:
    --------
    pandas.DataFrame
        DataFrame with hourly heart rate metrics
    """
    # Read the seconds data
    print("Reading heart rate data...")
    df = pd.read_csv(file_path)
    
    # Convert Time column to datetime
    df['Time'] = pd.to_datetime(df['Time'])
    
    # Extract hour from timestamp
    df['Hour'] = df['Time'].dt.floor('H')
    
    # Group by Id and Hour
    hourly_heartrate = df.groupby(['Id', 'Hour']).agg(
        avg_heartrate=('Value', 'mean'),
        max_heartrate=('Value', 'max'),
        min_heartrate=('Value', 'min'),
        resting_heartrate=('Value', lambda x: sorted(x)[:int(len(x)*0.1)]), # lowest 10% of readings
        readings_count=('Value', 'count'),
        heartrate_std=('Value', 'std')  # Standard deviation to measure variability
    ).reset_index()
    
    # Calculate true resting heart rate (average of lowest 10%)
    hourly_heartrate['resting_heartrate'] = hourly_heartrate['resting_heartrate'].apply(
        lambda x: sum(x)/len(x) if len(x) > 0 else 0
    )
    
    # Calculate heart rate zones (percentage of time in each zone)
    def calculate_zones(group):
        total = len(group)
        return pd.DataFrame([{
            'pct_low_hr': (group < 100).sum() / total * 100,
            'pct_moderate_hr': ((group >= 100) & (group < 140)).sum() / total * 100,
            'pct_high_hr': (group >= 140).sum() / total * 100
        }])
    
    # Calculate heart rate zones for each hour
    zones = df.groupby(['Id', 'Hour'])['Value'].apply(calculate_zones).reset_index(level=2, drop=True).reset_index()
    hourly_heartrate = hourly_heartrate.merge(zones, on=['Id', 'Hour'])
    
    # Round numeric columns to 2 decimal places
    numeric_columns = ['avg_heartrate', 'resting_heartrate', 'heartrate_std', 
                      'pct_low_hr', 'pct_moderate_hr', 'pct_high_hr']
    hourly_heartrate[numeric_columns] = hourly_heartrate[numeric_columns].round(2)
    
    # Format datetime for output
    hourly_heartrate['Hour'] = hourly_heartrate['Hour'].dt.strftime('%Y-%m-%d %H:00:00')
    
    # Save to CSV
    output_path = 'hourlyHeartrate_aggregated.csv'
    hourly_heartrate.to_csv(output_path, index=False)
    print(f"Successfully aggregated data for {len(hourly_heartrate)} hour-user combinations")
    print(f"Output saved to {os.path.abspath(output_path)}")
    
    return hourly_heartrate

# Example usage
if __name__ == "__main__":
    file_path = "/home/bshumway9/CS-4320/project/archive/4.12.16-5.12.16/Fitabase Data 4.12.16-5.12.16/heartrate_seconds_merged.csv"
    hourly_data = aggregate_heartrate_to_hourly(file_path)
    
    # Display summary statistics
    print("\nSummary statistics:")
    print(hourly_data[['avg_heartrate', 'max_heartrate', 'min_heartrate', 'resting_heartrate']].describe())
    
    # Show the first few rows
    print("\nSample of aggregated data:")
    print(hourly_data.head())

In [6]:
import pandas as pd
import os
from datetime import datetime

def aggregate_hourly_to_daily_heartrate(file_path='hourlyHeartrate_aggregated.csv'):
    """
    Aggregates hourly heart rate data into daily summaries.
    
    Parameters:
    -----------
    file_path : str
        Path to the hourly heart rate CSV file
        
    Returns:
    --------
    pandas.DataFrame
        DataFrame with daily heart rate metrics
    """
    # Read the hourly data
    print("Reading hourly heart rate data...")
    df = pd.read_csv(file_path)
    
    # Convert Hour to datetime
    df['Hour'] = pd.to_datetime(df['Hour'])
    df['Date'] = df['Hour'].dt.date
    
    # Group by Id and Date
    daily_heartrate = df.groupby(['Id', 'Date']).agg(
        avg_heartrate=('avg_heartrate', 'mean'),
        max_heartrate=('max_heartrate', 'max'),
        min_heartrate=('min_heartrate', 'min'),
        resting_heartrate=('resting_heartrate', 'mean'),
        total_readings=('readings_count', 'sum'),
        active_hours=('readings_count', 'count'),
        avg_heartrate_std=('heartrate_std', 'mean'),
        pct_time_low_hr=('pct_low_hr', 'mean'),
        pct_time_moderate_hr=('pct_moderate_hr', 'mean'),
        pct_time_high_hr=('pct_high_hr', 'mean')
    ).reset_index()
    
    # Format date as string in MM/DD/YYYY format
    daily_heartrate['Date'] = daily_heartrate['Date'].apply(
        lambda x: datetime.strftime(x, '%m/%d/%Y')
    )
    
    # Round numeric columns to 2 decimal places
    numeric_columns = ['avg_heartrate', 'resting_heartrate', 'avg_heartrate_std',
                      'pct_time_low_hr', 'pct_time_moderate_hr', 'pct_time_high_hr']
    daily_heartrate[numeric_columns] = daily_heartrate[numeric_columns].round(2)
    
    # Save to CSV
    output_path = 'dailyHeartrate_aggregated.csv'
    daily_heartrate.to_csv(output_path, index=False)
    print(f"Successfully aggregated data for {len(daily_heartrate)} day-user combinations")
    print(f"Output saved to {os.path.abspath(output_path)}")
    
    return daily_heartrate

# Example usage
if __name__ == "__main__":
    daily_data = aggregate_hourly_to_daily_heartrate("/home/bshumway9/CS-4320/project/archive/4.12.16-5.12.16/Fitabase Data 4.12.16-5.12.16/hourlyHeartrate_aggregated.csv")
    
    # Display summary statistics
    print("\nSummary statistics:")
    print(daily_data[['avg_heartrate', 'max_heartrate', 'min_heartrate', 
                      'resting_heartrate']].describe())
    
    # Show the first few rows
    print("\nSample of aggregated data:")
    print(daily_data.head())

In [15]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta
from sklearn.linear_model import LinearRegression

def impute_daily_weights(file_path='weightLogInfo_merged.csv', 
                        start_date='2016-03-12', 
                        end_date='2016-04-11'):
    """
    Imputes missing weight data for all users using linear regression.
    
    Parameters:
    -----------
    file_path : str
        Path to the weight log CSV file
    start_date : str
        Start date of the study period (YYYY-MM-DD)
    end_date : str
        End date of the study period (YYYY-MM-DD)
        
    Returns:
    --------
    pandas.DataFrame
        DataFrame with daily weights (original where available, imputed otherwise)
    """
    # Read the weight log data
    print("Reading weight log data from:", file_path)
    df = pd.read_csv(file_path)
    print(f"Found {len(df)} weight records for {df['Id'].nunique()} users")
    
    # Convert Date column to datetime - handle the correct format
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    # Extract just the date part (no time)
    df['Date'] = df['Date'].dt.floor('D')
    
    # Check if we have dates after conversion
    if df['Date'].isna().any():
        print("WARNING: Some dates could not be parsed correctly")
    
    # Print date range in data
    print(f"Date range in original data: {df['Date'].min()} to {df['Date'].max()}")
    
    # Create date range for all days in study period
    date_range = pd.date_range(start=start_date, end=end_date)
    
    # Initialize empty list for results
    all_weights = []
    
    # Process each user separately
    for user_id in df['Id'].unique():
        print(f"\nProcessing user ID: {user_id}")
        user_data = df[df['Id'] == user_id].copy()
        
        print(f"  Found {len(user_data)} weight measurements")
        print(f"  Date range: {user_data['Date'].min()} to {user_data['Date'].max()}")
        
        # Create complete date range for this user
        user_dates = pd.DataFrame(date_range, columns=['Date'])
        user_dates['Id'] = user_id
        
        # Merge with actual readings
        # First, select only date columns we need
        user_data_subset = user_data[['Id', 'Date', 'WeightKg', 'WeightPounds', 'BMI']].copy()
        
        # Check for duplicate dates and keep the latest entry
        user_data_subset = user_data_subset.sort_values('Date').drop_duplicates(['Id', 'Date'], keep='last')
        
        # Now merge
        user_full = user_dates.merge(
            user_data_subset, 
            on=['Id', 'Date'], 
            how='left'
        )
        
        print(f"  After merge, have {user_full.shape[0]} days, with {user_full['WeightKg'].notna().sum()} weight measurements")
        
        # Store original values to calculate imputation rates later
        original_count = user_full['WeightKg'].notna().sum()
        
        if len(user_data) > 1:  # Need at least 2 points for regression
            print("  Using linear regression for imputation")
            # Create features for regression (days since study start)
            user_full['days_from_start'] = (user_full['Date'] - pd.to_datetime(start_date)).dt.days
            
            # Fit regression model for each metric
            for column in ['WeightKg', 'WeightPounds', 'BMI']:
                if user_full[column].notna().sum() > 1:  # Check if we have enough data
                    # Prepare regression data
                    has_data = user_full[column].notna()
                    X = user_full.loc[has_data, 'days_from_start'].values.reshape(-1, 1)
                    y = user_full.loc[has_data, column].values
                    
                    # Fit model
                    model = LinearRegression()
                    model.fit(X, y)
                    
                    # Predict for all days where data is missing
                    missing_data = user_full[column].isna()
                    if missing_data.any():
                        X_missing = user_full.loc[missing_data, 'days_from_start'].values.reshape(-1, 1)
                        predictions = model.predict(X_missing)
                        
                        # Fill missing values with predictions
                        user_full.loc[missing_data, column] = predictions
                    
                    # Calculate range for logging
                    pred_min = user_full.loc[missing_data, column].min() if missing_data.any() else 0
                    pred_max = user_full.loc[missing_data, column].max() if missing_data.any() else 0
                    print(f"    Imputed {column}: valid range {pred_min:.2f} to {pred_max:.2f}")
                    
                elif user_full[column].notna().sum() == 1:  # Only one measurement
                    # If only one measurement with this column, use it for all days
                    mean_val = user_full[column].mean()
                    user_full[column] = user_full[column].fillna(mean_val)
                    print(f"    Used constant {column}: {mean_val:.2f}")
        else:
            print("  Using constant imputation (only one measurement)")
            # If only one measurement, use it for all days
            for column in ['WeightKg', 'WeightPounds', 'BMI']:
                if not user_full[column].isna().all():  # Check if we have any data
                    mean_val = user_full[column].mean()
                    user_full[column] = user_full[column].fillna(mean_val)
                    print(f"    Used constant {column}: {mean_val:.2f}")
                else:
                    print(f"    No {column} data available")
        
        # Calculate and print imputation stats
        imputed_count = user_full['WeightKg'].notna().sum() - original_count
        print(f"  Imputed {imputed_count} days out of {len(user_full)} total days ({imputed_count/len(user_full)*100:.1f}%)")
        
        # Add a column to indicate if the value was imputed
        user_full['IsImputed'] = np.zeros(len(user_full))
        # Mark all rows without original data as imputed
        for column in ['WeightKg', 'WeightPounds', 'BMI']:
            original_has_data = user_data_subset.set_index(['Id', 'Date'])[column].notna()
            for idx, row in user_full.iterrows():
                if (row['Id'], row['Date']) not in original_has_data.index or not original_has_data.get((row['Id'], row['Date']), False):
                    if pd.notna(row[column]):
                        user_full.loc[idx, 'IsImputed'] = 1
        
        # Remove the temporary column
        user_full = user_full.drop(columns=['days_from_start'], errors='ignore')
        
        # Add to results
        all_weights.append(user_full)
    
    # Combine all users
    result = pd.concat(all_weights, ignore_index=True)
    
    # Format date as string
    result['Date'] = result['Date'].dt.strftime('%Y-%m-%d')
    
    # Round values to 2 decimal places
    for column in ['WeightKg', 'WeightPounds', 'BMI']:
        if column in result.columns:
            result[column] = result[column].round(2)
    
    # Save to CSV
    output_path = 'dailyWeight_complete.csv'
    result.to_csv(output_path, index=False)
    print(f"\nSuccessfully processed weights for {df['Id'].nunique()} users")
    print(f"Output saved to {os.path.abspath(output_path)}")
    
    return result

# Example usage
if __name__ == "__main__":
    import os
    file_path = "/home/bshumway9/CS-4320/project/archive/4.12.16-5.12.16/Fitabase Data 4.12.16-5.12.16/weightLogInfo_merged.csv"
    daily_weights = impute_daily_weights(file_path, 
                                          start_date='2016-04-12', 
                                          end_date='2016-05-12')
    
    # Display summary statistics
    print("\nSummary statistics:")
    print(daily_weights[['WeightKg', 'WeightPounds', 'BMI']].describe())
    
    # Count of original vs imputed
    imputed = daily_weights['IsImputed'].sum()
    original = len(daily_weights) - imputed
    print(f"\nOriginal values: {original} ({original/len(daily_weights)*100:.1f}%)")
    print(f"Imputed values: {imputed} ({imputed/len(daily_weights)*100:.1f}%)")
    
    # Show a sample of data for one user
    if len(daily_weights) > 0:
        sample_user = daily_weights['Id'].iloc[0]
        print(f"\nSample data for user {sample_user}:")
        user_data = daily_weights[daily_weights['Id'] == sample_user][['Date', 'WeightKg', 'IsImputed']]
        print(user_data.head(10))

In [4]:
import pandas as pd
import os
from datetime import datetime

def combine_daily_data(
    calories_path='dailyCalories_aggregated.csv',
    heartrate_path='dailyHeartrate_aggregated.csv',
    intensities_path='dailyIntensities_aggregated.csv',
    sleep_path='dailySleep_aggregated.csv',
    steps_path='dailySteps_aggregated.csv',
    weight_path='dailyWeight_complete.csv',
    output_path='combined_daily_data.csv'
):
    """
    Combines multiple daily fitness data files into a single comprehensive dataset
    by merging two files at a time to reduce memory usage.
    
    Parameters:
    -----------
    calories_path : str
        Path to daily calories CSV file
    heartrate_path : str
        Path to daily heart rate CSV file
    intensities_path : str
        Path to daily intensities CSV file
    sleep_path : str
        Path to daily sleep CSV file
    steps_path : str
        Path to daily steps CSV file
    weight_path : str
        Path to daily weight CSV file
    output_path : str
        Path for the output combined CSV file
        
    Returns:
    --------
    pandas.DataFrame
        Combined DataFrame with all daily metrics
    """
    def merge_two_files(base_df, new_file_path):
        try:
            new_df = pd.read_csv(new_file_path)
            if 'Date' in new_df.columns:
                new_df['Date'] = pd.to_datetime(new_df['Date']).dt.strftime('%Y-%m-%d')
            else:
                raise KeyError(f"The 'Date' column is missing in the file: {new_file_path}")
            
            if 'Id' not in new_df.columns:
                raise KeyError(f"The 'Id' column is missing in the file: {new_file_path}")
            
            merged_df = base_df.merge(new_df, on=['Id', 'Date'], how='left')
            print(f"Merged data from {new_file_path}. Current shape: {merged_df.shape}")
            return merged_df
        except Exception as e:
            print(f"Error processing file {new_file_path}: {e}")
            return base_df

    # Start with the first file
    try:
        base_df = pd.read_csv(calories_path)
        if 'Date' in base_df.columns:
            base_df['Date'] = pd.to_datetime(base_df['Date']).dt.strftime('%Y-%m-%d')
        else:
            raise KeyError(f"The 'Date' column is missing in the file: {calories_path}")
        print(f"Loaded {len(base_df)} records from calories file")
    except Exception as e:
        print(f"Error loading calories file: {e}")
        return None

    # Merge files one by one
    base_df = merge_two_files(base_df, heartrate_path)
    base_df = merge_two_files(base_df, intensities_path)
    base_df = merge_two_files(base_df, sleep_path)
    base_df = merge_two_files(base_df, steps_path)
    base_df = merge_two_files(base_df, weight_path)

    # Save to CSV
    base_df.to_csv(output_path, index=False)
    print(f"Successfully combined data into {os.path.abspath(output_path)}")
    print(f"Final dataset has {len(base_df)} rows and {len(base_df.columns)} columns")
    
    # Display column names
    print("\nColumns in combined dataset:")
    print(base_df.columns.tolist())
    
    return base_df

# Example usage
if __name__ == "__main__":
    base_path = "/home/bshumway9/CS-4320/project/archive/3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/"
    
    combined_data = combine_daily_data(
        calories_path=base_path + "dailyCalories_aggregated.csv",
        heartrate_path=base_path + "dailyHeartrate_aggregated.csv",
        intensities_path=base_path + "dailyHighIntensities_aggregated.csv",
        sleep_path=base_path + "dailySleep_aggregated.csv",
        steps_path=base_path + "dailySteps_aggregated.csv",
        weight_path=base_path + "dailyWeight_complete.csv",
        output_path=base_path + "combined_daily_data.csv"
    )
    
    # Display a preview
    print("\nPreview of combined data:")
    print(combined_data.head())
    
    # Check for any issues with the merge
    print("\nColumns with missing data:")
    missing_counts = combined_data.isna().sum()
    print(missing_counts[missing_counts > 0])


Loaded 1021 records from calories file
Merged data from /home/bshumway9/CS-4320/project/archive/3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/dailyHeartrate_aggregated.csv. Current shape: (1021, 13)
Merged data from /home/bshumway9/CS-4320/project/archive/3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/dailyHighIntensities_aggregated.csv. Current shape: (1021, 14)
Merged data from /home/bshumway9/CS-4320/project/archive/3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/dailySleep_aggregated.csv. Current shape: (1021, 18)
Merged data from /home/bshumway9/CS-4320/project/archive/3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/dailySteps_aggregated.csv. Current shape: (1021, 23)
Merged data from /home/bshumway9/CS-4320/project/archive/3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/dailyWeight_complete.csv. Current shape: (1021, 27)
Successfully combined data into /home/bshumway9/CS-4320/project/archive/3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/combined_daily_data.csv
Final dataset has 1021 row

In [14]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

def convert_gym_to_fitbit_format(
    input_file='gym_members_exercise_tracking.csv',
    output_file='gym_members_fitbit_format.csv',
    reference_date="2023-01-01"
):
    """
    Converts gym members exercise tracking data to a format similar to Fitbit data.
    Creates a single accurate record per user rather than multiple days with variation.
    
    Parameters:
    -----------
    input_file : str
        Path to the gym members exercise tracking CSV file
    output_file : str
        Path for the output converted CSV file
    reference_date : str
        Date to use for all records (YYYY-MM-DD)
        
    Returns:
    --------
    pandas.DataFrame
        DataFrame with converted data in Fitbit-like format
    """
    print(f"Reading gym member data from {input_file}")
    
    # Read the gym members data
    gym_df = pd.read_csv(input_file)
    print(f"Found {len(gym_df)} gym member records")
    
    # Set date
    reference_date = pd.to_datetime(reference_date)
    
    # Create a list to hold all converted records
    all_records = []
    
    # Generate a unique ID for each gym member
    member_ids = [int(f"10{i}0000000") for i in range(1, len(gym_df) + 1)]
    
    print(f"Converting data for each member to Fitbit format for date {reference_date.strftime('%Y-%m-%d')}")
    
    # Process each gym member
    for idx, (_, member) in enumerate(gym_df.iterrows()):
        member_id = member_ids[idx]
        
        # Map workout type to activity intensity
        intensity_map = {
            'HIIT': 500,
            'Cardio': 400,
            'Strength': 350,
            'Yoga': 250
        }
        base_intensity = intensity_map.get(member['Workout_Type'], 300)
        
        # Create a record for this member
        record = {
            'Id': member_id,
            'Date': reference_date.strftime('%Y-%m-%d'),
            
            # Calories - direct mapping
            'Calories': int(member['Calories_Burned']),
            
            # Heart rate metrics - direct mapping
            'avg_heartrate': member['Avg_BPM'],
            'max_heartrate': member['Max_BPM'],
            'min_heartrate': member['Resting_BPM'],
            'resting_heartrate': member['Resting_BPM'],
            'total_readings': 1,  # Readings every 5 seconds
            'active_hours': member['Session_Duration (hours)'],
            'avg_heartrate_std': float(round((member['Max_BPM'] - member['Resting_BPM'])/10, 2)),  # Calculated from range
            
            # Heart rate zones - based on workout type and experience
            'pct_time_low_hr': max(0, 100 - member['Workout_Frequency (days/week)']*10 - member['Experience_Level']*10),
            'pct_time_moderate_hr': min(95, member['Workout_Frequency (days/week)']*10 + member['Experience_Level']*5),
            'pct_time_high_hr': member['Experience_Level'] * 5,
            
            # Intensity metrics - based on workout type
            'TotalIntensity': int(base_intensity),
            'AverageIntensity': round(base_intensity / (24 / member['Session_Duration (hours)']), 2),
            'HoursActive': member['Session_Duration (hours)'],
            'MaxIntensity': int(base_intensity * 1.5),  # Peak intensity
            'MaxIntensityHour': None,  # Default mid-morning workout time
            
            # Sleep metrics - reasonable estimates based on fitness level
            'sleep_minutes': None,
            'avg_sleep_quality': None,
            'sleep_records': None,
            'sleep_duration': None,  # Will fill in below
            
            # Steps metrics - calculated from calories and activity type
            'TotalSteps': None,  # Approximation based on calories
            'MaxStepsMinute': None,  # Age-adjusted max pace
            'ActiveMinutes': int(member['Session_Duration (hours)'] * 60),
            'AvgStepsPerActiveMinute': None,  # Will calculate below
            'PeakActivityTime': None,  # Default time
            
            # Weight metrics - direct mapping
            'WeightKg': member['Weight (kg)'],
            'WeightPounds': round(member['Weight (kg)'] * 2.20462, 2),
            'BMI': member['BMI'],
            'IsImputed': 0
        }
        
        # Calculate AvgStepsPerActiveMinute
        # if record['ActiveMinutes'] > 0:
        #     record['AvgStepsPerActiveMinute'] = round(record['TotalSteps'] / record['ActiveMinutes'], 2)
        
        # Format sleep duration
        # hours = record['sleep_minutes'] // 60
        # minutes = record['sleep_minutes'] % 60
        # record['sleep_duration'] = f"{hours}h {minutes}m"
        
        all_records.append(record)
    
    # Convert to DataFrame
    result_df = pd.DataFrame(all_records)
    
    # Sort by Id
    result_df = result_df.sort_values(['Id'])
    
    # Save to CSV
    result_df.to_csv(output_file, index=False)
    print(f"Converted data saved to {os.path.abspath(output_file)}")
    print(f"Generated {len(result_df)} records for {len(member_ids)} members")
    
    return result_df

# Example usage
if __name__ == "__main__":
    # Use a fixed date
    reference_date = "2023-01-01"
    converted_data = convert_gym_to_fitbit_format(
        input_file="/home/bshumway9/CS-4320/project/gym_members_exercise_tracking.csv",
        output_file="gym_members_fitbit_format.csv",
        reference_date=reference_date
    )
    
    # Display a preview
    print("\nPreview of converted data:")
    print(converted_data.head())
    
    # Summary statistics
    print("\nSummary statistics for key metrics:")
    metrics = ['Calories', 'avg_heartrate', 'TotalSteps', 'WeightKg', 'BMI', 'sleep_minutes']
    print(converted_data[metrics].describe())

Reading gym member data from /home/bshumway9/CS-4320/project/gym_members_exercise_tracking.csv
Found 973 gym member records
Converting data for each member to Fitbit format for date 2023-01-01
Converted data saved to /home/bshumway9/CS-4320/project/gym_members_fitbit_format.csv
Generated 973 records for 973 members

Preview of converted data:
           Id        Date  Calories  avg_heartrate  max_heartrate  \
0  1010000000  2023-01-01      1313            157            180   
1  1020000000  2023-01-01       883            151            179   
2  1030000000  2023-01-01       677            122            167   
3  1040000000  2023-01-01       532            164            190   
4  1050000000  2023-01-01       556            158            188   

   min_heartrate  resting_heartrate  total_readings  active_hours  \
0             60                 60               1          1.69   
1             66                 66               1          1.30   
2             54                 

In [5]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta

def convert_sleep_health_to_fitbit_format(
    input_file='Sleep_health_and_lifestyle_dataset.csv',
    output_file='sleep_health_fitbit_format.csv',
    reference_date=None
):
    """
    Converts sleep health and lifestyle dataset into Fitbit format matching combined_daily_data.csv.
    
    Parameters:
    -----------
    input_file : str
        Path to the sleep health dataset CSV file
    output_file : str
        Path for the output converted CSV file
    reference_date : str
        Start date to use for records (YYYY-MM-DD). If None, uses current date.
        
    Returns:
    --------
    pandas.DataFrame
        DataFrame with converted data in Fitbit-like format
    """
    print(f"Reading sleep health data from {input_file}")
    
    # Read the sleep health data
    sleep_df = pd.read_csv(input_file)
    print(f"Found {len(sleep_df)} person records")
    
    # Set reference date if not provided
    if reference_date is None:
        reference_date = datetime.now().strftime("%Y-%m-%d")
    
    reference_date = pd.to_datetime(reference_date)
    
    # Map BMI categories to numeric values
    bmi_category_map = {
        'Underweight': 18.0,
        'Normal': 22.0,
        'Normal Weight': 22.0,
        'Overweight': 27.5,
        'Obese': 32.5
    }
    
    # Create a list to hold all converted records
    all_records = []
    
    # Process each person
    for idx, person in sleep_df.iterrows():
        person_id = int(f"5{person['Person ID']:08d}")
        
        # Process physical activity level as intensity
        activity_level = person['Physical Activity Level']
        
        # Convert stress level to intensity inversely (lower stress = higher activity)
        stress_level = person['Stress Level']
        
        # Calculate BMI from category if available
        if pd.notna(person['BMI Category']):
            bmi = bmi_category_map.get(person['BMI Category'], 22.0)
        else:
            bmi = 22.0  # Default value
            
        # Parse blood pressure
        bp = person['Blood Pressure'].split('/') if pd.notna(person['Blood Pressure']) else [120, 80]
        systolic = int(bp[0])
        diastolic = int(bp[1])
        
        # Calculate sleep minutes from hours
        sleep_hours = person['Sleep Duration']
        sleep_minutes = int(sleep_hours * 60)
        
        # Format sleep duration as hours and minutes
        sleep_hours_int = int(sleep_hours)
        sleep_minutes_remainder = int((sleep_hours - sleep_hours_int) * 60)
        sleep_duration = f"{sleep_hours_int}h {sleep_minutes_remainder}m"
        
        # Create record
        record = {
            'Id': person_id,
            'Date': reference_date.strftime('%Y-%m-%d'),
            
            # Calories - estimated based on activity level and BMI
            'Calories': int(1500 + (activity_level * 8)),
            
            # Heart rate metrics
            'avg_heartrate': person['Heart Rate'],
            'max_heartrate': int(person['Heart Rate'] * 1.5),
            'min_heartrate': int(person['Heart Rate'] * 0.8),
            'resting_heartrate': int(person['Heart Rate'] * 0.9),
            'total_readings': int(activity_level * 100),
            'active_hours': activity_level / 60,  # Assuming activity level is in minutes
            'avg_heartrate_std': 10.0,  # Default value
            
            # Heart rate zones - estimated from activity and stress
            'pct_time_low_hr': 100 - (activity_level / 2),
            'pct_time_moderate_hr': (activity_level / 2) - (activity_level / 10),
            'pct_time_high_hr': activity_level / 10,
            
            # Intensity metrics - based on activity level
            'TotalIntensity': activity_level,
            # 'AverageIntensity': activity_level / 24,
            # 'HoursActive': activity_level / 60,
            # 'MaxIntensity': int(activity_level * 0.2),
            # 'MaxIntensityHour': "10:00 AM",
            
            # Sleep metrics - direct mapping
            'sleep_minutes': sleep_minutes,
            'avg_sleep_quality': person['Quality of Sleep'] / 5,  # Scale to match fitbit format
            'sleep_records': 1,
            'sleep_duration': sleep_duration,
            
            # Steps metrics - direct mapping
            'TotalSteps': person['Daily Steps'],
            'MaxStepsMinute': int(person['Daily Steps'] / 120),  # Estimated peak rate
            'ActiveMinutes': int(activity_level),
            'AvgStepsPerActiveMinute': round(person['Daily Steps'] / max(1, activity_level), 2),
            'PeakActivityTime': "10:30",  # Default time
            
            # Weight metrics - estimated from BMI
            'WeightKg': round(bmi * 3, 1),  # Rough estimate
            'WeightPounds': round(bmi * 3 * 2.20462, 2),
            'BMI': bmi,
            'IsImputed': 0
        }
        
        all_records.append(record)
    
    # Convert to DataFrame
    result_df = pd.DataFrame(all_records)
    
    # Sort by Id and Date
    result_df = result_df.sort_values(['Id', 'Date'])
    
    # Save to CSV
    result_df.to_csv(output_file, index=False)
    print(f"Converted data saved to {os.path.abspath(output_file)}")
    print(f"Generated {len(result_df)} records from {len(sleep_df)} people")
    
    return result_df

# Example usage
if __name__ == "__main__":
    # Set a fixed reference date
    reference_date = "2023-01-01"
    converted_data = convert_sleep_health_to_fitbit_format(
        input_file="/home/bshumway9/CS-4320/project/Sleep_health_and_lifestyle_dataset.csv",
        output_file="sleep_health_fitbit_format.csv",
        reference_date=reference_date
    )
    
    # Display a preview
    print("\nPreview of converted data:")
    print(converted_data.head())

Reading sleep health data from /home/bshumway9/CS-4320/project/Sleep_health_and_lifestyle_dataset.csv
Found 374 person records
Converted data saved to /home/bshumway9/CS-4320/project/sleep_health_fitbit_format.csv
Generated 374 records from 374 people

Preview of converted data:
          Id        Date  Calories  avg_heartrate  max_heartrate  \
0  500000001  2023-01-01      1836             77            115   
1  500000002  2023-01-01      1980             75            112   
2  500000003  2023-01-01      1980             75            112   
3  500000004  2023-01-01      1740             85            127   
4  500000005  2023-01-01      1740             85            127   

   min_heartrate  resting_heartrate  total_readings  active_hours  \
0             61                 69            4200           0.7   
1             60                 67            6000           1.0   
2             60                 67            6000           1.0   
3             68                 76

In [7]:
import pandas as pd

def filter_resting_heartrate(input_file, output_file):
    """
    Filters rows without a resting_heartrate from the input CSV file and saves the result to a new file.
    
    Parameters:
    -----------
    input_file : str
        Path to the input CSV file.
    output_file : str
        Path to save the filtered CSV file.
    """
    # Read the input file
    df = pd.read_csv(input_file)
    
    # Filter rows where resting_heartrate is not null
    filtered_df = df[df['resting_heartrate'].notna()]
    
    # Save the filtered DataFrame to a new CSV file
    filtered_df.to_csv(output_file, index=False)
    print(f"Filtered data saved to {output_file}")

# Example usage
input_file = '/home/bshumway9/CS-4320/project/archive/3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/combined_daily_data.csv'
output_file = '/home/bshumway9/CS-4320/project/archive/3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/filtered_combined_daily_data.csv'
filter_resting_heartrate(input_file, output_file)

Filtered data saved to /home/bshumway9/CS-4320/project/archive/3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/filtered_combined_daily_data.csv


In [8]:
import pandas as pd
import os
from datetime import datetime

def combine_csv_files(file_paths, output_path='combined_data.csv', preserve_all_columns=True):
    """
    Combines multiple CSV files with similar structure into a single CSV file.
    
    Parameters:
    -----------
    file_paths : list
        List of file paths to the CSV files to combine
    output_path : str
        Path for the output combined CSV file
    preserve_all_columns : bool
        If True, keeps all columns from all files. If False, only keeps columns that appear in all files.
        
    Returns:
    --------
    pandas.DataFrame
        Combined DataFrame
    """
    if not file_paths:
        raise ValueError("No file paths provided")
    
    print(f"Starting to combine {len(file_paths)} CSV files...")
    
    # Initialize empty list to store dataframes
    all_dfs = []
    total_rows = 0
    
    # Process each file
    for i, file_path in enumerate(file_paths):
        try:
            # Read the CSV file
            df = pd.read_csv(file_path)
            rows = len(df)
            total_rows += rows
            print(f"Loaded file {i+1}/{len(file_paths)}: {os.path.basename(file_path)} - {rows} rows")
            
            # Add source file info if requested
            df['source_file'] = os.path.basename(file_path)
            
            # Store dataframe in the list
            all_dfs.append(df)
        except Exception as e:
            print(f"Error loading file {file_path}: {e}")
    
    if not all_dfs:
        raise ValueError("No valid CSV files were loaded")
    
    # Combine all dataframes
    if preserve_all_columns:
        # Use concatenation to preserve all columns (will create NaN values for missing columns)
        combined_df = pd.concat(all_dfs, ignore_index=True, sort=False)
        print(f"Combined dataframes with all columns preserved")
    else:
        # Find common columns across all dataframes
        common_columns = set(all_dfs[0].columns)
        for df in all_dfs[1:]:
            common_columns = common_columns.intersection(set(df.columns))
        
        common_columns = sorted(list(common_columns))
        print(f"Found {len(common_columns)} common columns across all files")
        
        # Use only common columns
        filtered_dfs = [df[common_columns] for df in all_dfs]
        combined_df = pd.concat(filtered_dfs, ignore_index=True)
    
    # Save to output file
    combined_df.to_csv(output_path, index=False)
    print(f"Successfully combined {len(all_dfs)} files with {total_rows} total rows into {os.path.abspath(output_path)}")
    print(f"Final dataset has {len(combined_df)} rows and {len(combined_df.columns)} columns")
    
    return combined_df

# Example usage
if __name__ == "__main__":
    # Example file paths
    files_to_combine = [
        "/home/bshumway9/CS-4320/project/archive/3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/filtered_combined_daily_data.csv",
        "/home/bshumway9/CS-4320/project/sleep_health_fitbit_format.csv",
        # "/home/bshumway9/CS-4320/project/gym_members_fitbit_format.csv",
        "/home/bshumway9/CS-4320/project/archive/4.12.16-5.12.16/Fitabase Data 4.12.16-5.12.16/filtered_combined_daily_data.csv"
    ]
    
    # Combine files
    combined_data = combine_csv_files(
        file_paths=files_to_combine,
        output_path="/home/bshumway9/CS-4320/project/all_combined_data.csv",
        preserve_all_columns=True
    )
    
    # Display a summary of the combined data
    print("\nColumns in combined dataset:")
    print(combined_data.columns.tolist())
    
    # Check for any data type issues
    print("\nData types in combined dataset:")
    print(combined_data.dtypes)
    
    # Show a preview
    print("\nPreview of combined data:")
    print(combined_data.head())

Starting to combine 3 CSV files...
Loaded file 1/3: filtered_combined_daily_data.csv - 143 rows
Loaded file 2/3: sleep_health_fitbit_format.csv - 374 rows
Loaded file 3/3: filtered_combined_daily_data.csv - 334 rows
Combined dataframes with all columns preserved
Successfully combined 3 files with 851 total rows into /home/bshumway9/CS-4320/project/all_combined_data.csv
Final dataset has 851 rows and 28 columns

Columns in combined dataset:
['Id', 'Date', 'Calories', 'avg_heartrate', 'max_heartrate', 'min_heartrate', 'resting_heartrate', 'total_readings', 'active_hours', 'avg_heartrate_std', 'pct_time_low_hr', 'pct_time_moderate_hr', 'pct_time_high_hr', 'TotalIntensity', 'sleep_minutes', 'avg_sleep_quality', 'sleep_records', 'sleep_duration', 'TotalSteps', 'MaxStepsMinute', 'ActiveMinutes', 'AvgStepsPerActiveMinute', 'PeakActivityTime', 'WeightKg', 'WeightPounds', 'BMI', 'IsImputed', 'source_file']

Data types in combined dataset:
Id                           int64
Date                 